# Setup & Configuration

Packages are pre-installed in the `odra5-py312` virtual environment.  
Make sure you selected the **"Python 3.12 (Odra5)"** kernel before running.

In [1]:
print("chuj smierdzacy")

chuj smierdzacy


In [1]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info[:2] == (3, 12), f"Need Python 3.12, got {sys.version_info[:2]}. Select 'Python 3.12 (Odra5)' kernel."

import qiskit, iqm.qiskit_iqm, torch, sklearn, qiskit_machine_learning
print(f"qiskit:    {qiskit.__version__}")
print(f"torch:     {torch.__version__}")
print(f"sklearn:   {sklearn.__version__}")
print("All packages OK")

Python: 3.12.12 (main, Oct  9 2025, 11:07:00) [Clang 17.0.0 (clang-1700.4.4.1)]


/Users/jkw/Documents/uni/axion/QC1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


qiskit:    2.1.2
torch:     2.10.0
sklearn:   1.8.0
All packages OK


## Required Imports
All necessary libraries for quantum computing, machine learning, and data processing.

In [7]:
# IQM & Qiskit imports
from iqm.qiskit_iqm import IQMProvider
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator, PrimitiveResult, PubResult
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.primitives.containers.data_bin import DataBin
from qiskit.quantum_info import SparsePauliOp

# Qiskit Machine Learning
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN

# PyTorch
import torch
import torch.nn as nn

# Data science & ML
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, accuracy_score
from ucimlrepo import fetch_ucirepo

# Visualization
import matplotlib.pyplot as plt

# Utilities
import sys
import subprocess
import time


---
# Circuit Definitions

## Hardware-Efficient Ansatz
Custom quantum circuit designed for IQM's star topology (5 qubits, QB3 as central hub).

In [8]:
def ansatz(n_qubits, depth):
    """
    Constructs a hardware-efficient ansatz tailored for a star topology.
    QB3 (index 2) acts as the central hub for entanglement to avoid SWAP gates.
    Native CZ gates are used to minimize decomposition errors.
    """

    # Each full iteration (2 layers) consumes:
    # Layer 1: n_qubits (RY) + 4 (RZ before CZ) = 9 parameters
    # Layer 2: n_qubits (RX) + 4 (RY before CZ) = 9 parameters
    # Total = 18 parameters per iteration (where depth // 2 is the number of iterations)
    params_per_iter = 18
    total_params = params_per_iter * (depth // 2)
    theta = ParameterVector('θ', total_params)

    qc = QuantumCircuit(n_qubits)

    # The loop iterates (depth // 2) times to execute two-layer blocks.
    for j in range(depth // 2):
        offset = j * params_per_iter

        # -------- Layer 1: RY + Star CZ (RZ-based) --------

        # Sub-layer: Independent RY rotations on all qubits
        for i in range(n_qubits):
            qc.ry(theta[offset + i], i)

        # Sub-layer: Entanglement using Star Topology
        # QB3 (index 2) is the central qubit. We connect it to [0, 1, 3, 4].
        # RZ rotations are added to maintain expressibility while using native CZ.
        target_qubits = [0, 1, 3, 4]
        for idx, target in enumerate(target_qubits):
            # Using parameters offset+5 to offset+8
            qc.rz(theta[offset + n_qubits + idx], target)
            qc.cz(2, target)

        # -------- Layer 2: RX + Star CZ (RY-based) --------

        # Move the offset forward for the second layer within the same iteration
        offset_layer2 = offset + 9

        # Sub-layer: Independent RX rotations on all qubits
        for i in range(n_qubits):
            qc.rx(theta[offset_layer2 + i], i)

        # Sub-layer: Entanglement using Star Topology
        # RY rotations are used here to simulate the effect of a CRY-like interaction.
        for idx, target in enumerate(target_qubits):
            # Using parameters offset_layer2+5 to offset_layer2+8
            qc.ry(theta[offset_layer2 + n_qubits + idx], target)
            qc.cz(2, target)

    return qc

## IQM Backend Estimator
Custom estimator class that interfaces with IQM quantum hardware and tracks detailed timing information.

<details>
<summary>📊 Timing Metrics Captured</summary>

- **QPU Execution Time**: Actual quantum circuit execution
- **Compilation Time**: Circuit transpilation and optimization  
- **Queue Time**: Waiting for QPU availability
- **Network Time**: Job upload and result download
- **Total Job Time**: End-to-end execution

</details>

In [9]:
class SimpleIQMJob:
    def __init__(self, result):
        self._result = result
    
    def result(self):
        return self._result

class IQMBackendEstimator(BaseEstimatorV2):
    def __init__(self, backend, options=None):
        super().__init__()
        self._backend = backend
        self._options = options or {"shots": 100}
        self.timestamp_history = []
        self.total_qpu_time = 0.0

    def _extract_timestamps(self, result):
        try:
            timeline = result._metadata.get('timeline', [])
            if not timeline:
                return None
            timestamps = {}
            for entry in timeline:
                timestamps[entry.status] = entry.timestamp
            return timestamps
        except Exception:
            return None

    def run(self, pubs, precision=None):
        if not isinstance(pubs, list): pubs = [pubs]
        job_results = []
        
        base_circuit = pubs[0][0]
        circuit_with_meas = base_circuit.copy()
        if circuit_with_meas.num_clbits == 0:
            circuit_with_meas.measure_all()
        
        transpiled_qc = transpile(circuit_with_meas, self._backend, optimization_level=3)
        
        for pub in pubs:
            _, observables, parameter_values = pub
            if parameter_values.ndim == 1:
                parameter_values = [parameter_values]
            
            # Bind all circuits first
            all_bound = []
            for params in parameter_values:
                all_bound.append(transpiled_qc.assign_parameters(params))
            
            # Send ALL circuits in ONE job
            try:
                print(f"Sending batch of {len(all_bound)} circuits to QPU ({self._options['shots']} shots)...")
                job = self._backend.run(all_bound, shots=self._options["shots"])
                result = job.result()
                
                ts = self._extract_timestamps(result)
                if ts:
                    exec_start = ts.get('execution_started')
                    exec_end = ts.get('execution_ended')
                    comp_start = ts.get('compilation_started')
                    comp_end = ts.get('compilation_ended')
                    job_created = ts.get('created')
                    job_completed = ts.get('completed')
                    
                    if exec_start and exec_end:
                        execution_time = (exec_end - exec_start).total_seconds()
                        compile_time = (comp_end - comp_start).total_seconds() if comp_start and comp_end else 0
                        job_time = (job_completed - job_created).total_seconds() if job_created and job_completed else 0
                        
                        self.timestamp_history.append({
                            'execution_time_qpu': execution_time,
                            'job_time_total': job_time,
                            'compile_time': compile_time,
                            'raw_timestamps': ts
                        })
                        self.total_qpu_time += execution_time
                        
                        print(f"BATCH DONE - QPU: {execution_time*1000:.2f}ms | "
                              f"Compilation: {compile_time*1000:.2f}ms | "
                              f"Job total: {job_time:.3f}s")
                
                counts_list = result.get_counts()
                if not isinstance(counts_list, list):
                    counts_list = [counts_list]
                
                pub_expectations = []
                for counts in counts_list:
                    shots = sum(counts.values())
                    count_0 = 0
                    for bitstring, count in counts.items():
                        if bitstring[-1] == '0':
                            count_0 += count
                    p0 = count_0 / shots
                    p1 = 1 - p0
                    pub_expectations.append(p0 - p1)
                    
            except Exception as e:
                print(f"Batch job failed: {e}")
                pub_expectations = [0.0] * len(all_bound)
            
            data = DataBin(evs=np.array(pub_expectations), shape=(len(pub_expectations),))
            job_results.append(PubResult(data=data))

        return SimpleIQMJob(PrimitiveResult(job_results))
    
    def print_timing_summary(self):
        if not self.timestamp_history:
            print("No timing data.")
            return
        
        print("\n" + "="*60)
        print("TIMING SUMMARY")
        print("="*60)
        print(f"Number of batch jobs: {len(self.timestamp_history)}")
        
        for i, t in enumerate(self.timestamp_history):
            ts = t['raw_timestamps']
            qpu = (ts['execution_ended'] - ts['execution_started']).total_seconds() if ts.get('execution_started') and ts.get('execution_ended') else 0
            comp = (ts['compilation_ended'] - ts['compilation_started']).total_seconds() if ts.get('compilation_started') and ts.get('compilation_ended') else 0
            print(f"  Job {i+1}: QPU={qpu*1000:.2f}ms | Compile={comp*1000:.2f}ms | Total={t['job_time_total']:.3f}s")
        
        total_qpu = sum(t['execution_time_qpu'] for t in self.timestamp_history)
        total_job = sum(t['job_time_total'] for t in self.timestamp_history)
        print(f"\nTotal QPU time: {total_qpu*1000:.2f}ms")
        print(f"Total job time: {total_job:.3f}s")
        print("="*60)

## Hybrid Quantum-Classical Model
PyTorch-based model that integrates quantum circuits with classical neural networks.

In [10]:
"""
    The code below constructs the class HybridModel. It is built using the Qiskit and Pytorch library and
    and utilizes its built-in tools, to create a model connecting classical and quantum computing.

"""

class HybridModel(nn.Module):
    def __init__(self, ansatz_circuit, num_qubits):
        super().__init__()
        self.feature_map = self.angle_encoding(num_qubits)

        # Connecting the quantum circuit. Connecting our feature map (data) and ansatz
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        # Firstly, we inicialize parameters. Our quantum model cannot tell whether the number came from ansatz or feature.
        # That is why here we sort them into two lists. If the number came from feature_map, then it will be a feature and the other way around.
        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)

        '''
        Measure the Z-operator (spin) on the very first qubit (q_0) and ignore all the other qubits.
        Qiskit reads the string in a reversed order, that is why the Z gate is on the end.
        SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)]) converts string into a mathematical matrix that Qiskit can use for calculations
        Coefficient = 1 is a weight we multiply our result by. In QML it is mostly set to 1
        '''

        observable = SparsePauliOp.from_list([("I" * (num_qubits - 1) + "Z", 1)])

        # Estimator takes ansatz, observables and parameters (data and weights), returns the Expectation value.
        # !!!! CHANGE WHEN USING ON QUANTUM COMPUTER
        # Needed when running quantum simulations, it should be changed when implementing on real quantum computer
        estimator = StatevectorEstimator(seed=42)

        # Compute the gradients of the sampling probability by the Parameter Shift Rule.
        gradient = ParamShiftEstimatorGradient(estimator)


        '''
        The EstimatorQNN
        This class from Qiskit Machine Learning is used to instantiate the quantum neural network.
        It leverages the Qiskit Primitives (Estimator) to efficiently calculate expectation values
        of the quantum circuit. This allows the model to output continuous, differentiable values (gradients)
        required for backpropagation in hybrid quantum-classical training.
        '''

        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient
        )

        '''
        TORCH CONNECTOR
        This line initializes the TorchConnector, which serves as a bridge between Qiskit and PyTorch. It wraps the Quantum Neural Network (QNN)
        to make it function as a standard, differentiable PyTorch module (nn.Module).
        This integration allows the quantum parameters to be optimized using standard PyTorch tools like
        the Adam optimizer and automatic differentiation.
        '''
        self.quantum_layer = TorchConnector(self.qnn)

        """
        Creates a Feature Map circuit using Angle Encoding. It maps classical input vectors
        to the quantum space by applying Ry(theta) rotations on each qubit,
        where the rotation angle theta corresponds to the input feature value.
        This effectively encodes the data into the amplitudes of the quantum state
        """

    def angle_encoding(self, num_qubits):
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector('x', num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    '''
    This function acts as the main execution path. When the model receives data,
    the forward function passes it into the quantum layer to be processed.
    The quantum layer calculates the result based on the current circuit parameters and returns the prediction.
    '''
    def forward(self, x):
        return self.quantum_layer(x)

---
# Utility Functions

## Helper Functions for Evaluation

<details>
<summary>🔧 Click to view helper function definitions</summary>

The following helper functions simplify common operations:
- **`connect_to_iqm_backend()`**: Manages IQM connection  
- **`load_depth2_data()`**: Loads test data from CSV
- **`load_depth2_weights()`**: Loads pretrained model weights

</details>

In [11]:
# ========================================================
# HELPER FUNCTIONS FOR EVALUATION
# ========================================================

def connect_to_iqm_backend():
    global iqm_backend

    if 'iqm_backend' not in globals() or iqm_backend is None:
        print("\nIQM backend not found. Connecting...")
        try:
            base_url = "https://odra5.e-science.pl/"
            token = input("Enter IQM Token: ")

            provider = IQMProvider(base_url, token=token)
            iqm_backend = provider.get_backend()

            print(f"Connected to backend: {iqm_backend.name}")
        except Exception as e:
            print(f"Connection error: {e}")
            raise RuntimeError("Failed to connect to IQM backend") from e
    else:
        print(f"Using existing IQM backend: {iqm_backend.name}")

        try:
            client = getattr(iqm_backend, "client", None)
            server_client = getattr(client, "_iqm_server_client", None) if client is not None else None
            headers = getattr(server_client, "_default_headers", None) if server_client is not None else None

            auth_present = None
            auth_len = None
            if isinstance(headers, dict) and "Authorization" in headers and headers["Authorization"] is not None:
                auth_present = True
                auth_len = len(str(headers["Authorization"]))
            elif isinstance(headers, dict):
                auth_present = False

            try:
                _ = client.get_dynamic_quantum_architecture()
            except TypeError:
                _ = client.get_dynamic_quantum_architecture(None)

        except Exception:
            iqm_backend = None
            return connect_to_iqm_backend()

    return iqm_backend


def load_depth2_data():
    global X_test_depth2, y_test_depth2
    
    if 'X_test_depth2' not in globals():
        try:
            depth2_data = np.loadtxt("depth2/testset.csv", delimiter=",", skiprows=1)
            
            if depth2_data.shape[1] != 6:
                raise ValueError(f"Expected 6 columns, got {depth2_data.shape[1]}")
            
            X_test_depth2 = depth2_data[:, :5].astype(np.float32)
            y_test_depth2 = depth2_data[:, 5].astype(np.float32)
            
            print(f"Loaded depth2 testset: {X_test_depth2.shape[0]} samples")
        except Exception as e:
            print(f"Error loading testset: {e}")
            raise
    else:
        print(f"Using existing depth2 testset: {X_test_depth2.shape[0]} samples")
    
    return X_test_depth2, y_test_depth2


def load_depth2_weights(model, strip_prefix=False):
    try:
        loaded_state = torch.load("depth2/model_checkpoint_epoch_22.pth", weights_only=True)
        
        if strip_prefix:
            adjusted_state = {}
            for key, value in loaded_state.items():
                if key.startswith("quantum_layer."):
                    new_key = key.replace("quantum_layer.", "", 1)
                    adjusted_state[new_key] = value
                else:
                    adjusted_state[key] = value
            loaded_state = adjusted_state
        
        model.load_state_dict(loaded_state)
        print("Loaded weights from depth2/model_checkpoint_epoch_22.pth")
        
    except Exception as e:
        print(f"Error loading weights: {e}")
        raise

print("Helper functions defined")

Helper functions defined


---
# Model Setup

## Initialize Quantum Model
Create a 5-qubit model with depth-2 ansatz for evaluation.

In [12]:
# Initializing the model with 5 qubits
num_qubits = 5
final_ansatz = ansatz(num_qubits, 2)
model = HybridModel(final_ansatz, num_qubits)

---
# Evaluation Experiments

## Shot Count Comparison Analysis
Compare model accuracy and performance across different shot counts on IQM hardware vs StatevectorEstimator baseline.

**What this cell does:**
1. Prompts for shot counts and sample size
2. Evaluates baseline with StatevectorEstimator  
3. Tests multiple shot configurations on IQM hardware
4. Generates comparison graphs
5. Displays results summary table

In [19]:
# ========================================================
# 10-Run Stability Test (Automated)
# 500 shots, 275 samples, 10 runs - no manual input
# ========================================================

N_RUNS = 10
N_SHOTS = int(input("How many shots?: "))
N_SAMPLES = 275

print("=" * 80)
print(f"AUTOMATED STABILITY TEST: {N_RUNS} runs x {N_SHOTS} shots x {N_SAMPLES} samples")
print("=" * 80)

# ========================================================
# A. Data Preparation
# ========================================================

X_test_depth2_full, y_test_depth2_full = load_depth2_data()

if N_SAMPLES < len(X_test_depth2_full):
    np.random.seed(42)
    indices = np.random.choice(len(X_test_depth2_full), size=N_SAMPLES, replace=False)
    X_sub = X_test_depth2_full[indices]
    y_sub = y_test_depth2_full[indices]
else:
    X_sub = X_test_depth2_full
    y_sub = y_test_depth2_full

X_tensor = torch.tensor(X_sub, dtype=torch.float32)
print(f"Using {len(X_sub)} samples\n")

# ========================================================
# B. Connect to IQM
# ========================================================

iqm_backend = connect_to_iqm_backend()
print()

# ========================================================
# C. Run 10x (Statevector + Hardware each time)
# ========================================================

all_runs = []

for run_idx in range(1, N_RUNS + 1):
    print(f"--- Run {run_idx}/{N_RUNS} ---")
    
    # Statevector (fresh model each run)
    torch.manual_seed(42)
    sv_ansatz = ansatz(5, 2)
    sv_model = HybridModel(sv_ansatz, 5)
    load_depth2_weights(sv_model, strip_prefix=False)
    sv_model.eval()
    
    start_time = time.time()
    with torch.no_grad():
        sv_preds = sv_model(X_tensor).numpy().flatten()
    sv_time = (time.time() - start_time) / N_SAMPLES
    
    sv_labels = np.where(sv_preds > 0, 1, -1)
    sv_acc = accuracy_score(y_sub, sv_labels)
    sv_f1 = f1_score(y_sub, sv_labels, pos_label=1)
    
    # Hardware
    hw_estimator = IQMBackendEstimator(iqm_backend, options={"shots": N_SHOTS})
    
    hw_ansatz = ansatz(5, 2)
    hw_fm = HybridModel(hw_ansatz, 5).angle_encoding(5)
    
    hw_qc = QuantumCircuit(5)
    hw_qc.compose(hw_fm, qubits=range(5), inplace=True)
    hw_qc.compose(hw_ansatz, inplace=True)
    
    observable = SparsePauliOp.from_list([("I" * 4 + "Z", 1)])
    
    hw_qnn = EstimatorQNN(
        circuit=hw_qc,
        observables=observable,
        input_params=list(hw_fm.parameters),
        weight_params=list(hw_ansatz.parameters),
        estimator=hw_estimator
    )
    
    hw_model = TorchConnector(hw_qnn)
    load_depth2_weights(hw_model, strip_prefix=True)
    
    with torch.no_grad():
        hw_preds = hw_model(X_tensor).numpy().flatten()
    
    hw_labels = np.where(hw_preds > 0, 1, -1)
    hw_acc = accuracy_score(y_sub, hw_labels)
    hw_f1 = f1_score(y_sub, hw_labels, pos_label=1)
    hw_qpu = hw_estimator.total_qpu_time / N_SAMPLES
    
    all_runs.append({
        'sv_acc': sv_acc, 'sv_f1': sv_f1, 'sv_time': sv_time,
        'hw_acc': hw_acc, 'hw_f1': hw_f1, 'qpu_time': hw_qpu
    })
    print(f"  SV: Acc={sv_acc:.4f} F1={sv_f1:.4f} | Odra: Acc={hw_acc:.4f} F1={hw_f1:.4f} QPU/s={hw_qpu:.4f}s\n")

# ========================================================
# D. Summary Table
# ========================================================

sv_accs = [r['sv_acc'] for r in all_runs]
sv_f1s = [r['sv_f1'] for r in all_runs]
hw_accs = [r['hw_acc'] for r in all_runs]
hw_f1s = [r['hw_f1'] for r in all_runs]
qpus = [r['qpu_time'] for r in all_runs]

print("\n" + "=" * 100)
print(f"RESULTS: {N_RUNS} RUNS x {N_SHOTS} SHOTS x {N_SAMPLES} SAMPLES")
print("=" * 100)
print(f"{'Run':<6} | {'SV Acc':<10} | {'SV F1':<10} | {'Odra Acc':<10} | {'Odra F1':<10} | {'QPU Time/sample (s)'}")
print("-" * 100)
for i, r in enumerate(all_runs):
    print(f"{i+1:<6} | {r['sv_acc']:<10.4f} | {r['sv_f1']:<10.4f} | {r['hw_acc']:<10.4f} | {r['hw_f1']:<10.4f} | {r['qpu_time']:.4f}")
print("-" * 100)
print(f"{'Mean':<6} | {np.mean(sv_accs):<10.4f} | {np.mean(sv_f1s):<10.4f} | {np.mean(hw_accs):<10.4f} | {np.mean(hw_f1s):<10.4f} | {np.mean(qpus):.4f}")
print(f"{'Std':<6} | {np.std(sv_accs):<10.4f} | {np.std(sv_f1s):<10.4f} | {np.std(hw_accs):<10.4f} | {np.std(hw_f1s):<10.4f} | {np.std(qpus):.4f}")
print("=" * 100)

How many shots?:  5000


AUTOMATED STABILITY TEST: 10 runs x 5000 shots x 275 samples
Using existing depth2 testset: 275 samples
Using 275 samples

Using existing IQM backend: IQMBackend

--- Run 1/10 ---
Loaded weights from depth2/model_checkpoint_epoch_22.pth


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Loaded weights from depth2/model_checkpoint_epoch_22.pth
Sending batch of 275 circuits to QPU (5000 shots)...


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


...

Loaded weights from depth2/model_checkpoint_epoch_22.pth
Sending batch of 275 circuits to QPU (5000 shots)...
BATCH DONE - QPU: 553383.74ms | Compilation: 868.07ms | Job total: 556.695s
  SV: Acc=0.8400 F1=0.8268 | Odra: Acc=0.8509 F1=0.8476 QPU/s=2.0123s

--- Run 9/10 ---
Loaded weights from depth2/model_checkpoint_epoch_22.pth


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Loaded weights from depth2/model_checkpoint_epoch_22.pth
Sending batch of 275 circuits to QPU (5000 shots)...


No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


BATCH DONE - QPU: 553427.16ms | Compilation: 912.43ms | Job total: 556.559s
  SV: Acc=0.8400 F1=0.8268 | Odra: Acc=0.8473 F1=0.8433 QPU/s=2.0125s

--- Run 10/10 ---
Loaded weights from depth2/model_checkpoint_epoch_22.pth
Loaded weights from depth2/model_checkpoint_epoch_22.pth
Sending batch of 275 circuits to QPU (5000 shots)...
BATCH DONE - QPU: 553430.14ms | Compilation: 508.61ms | Job total: 557.095s
  SV: Acc=0.8400 F1=0.8268 | Odra: Acc=0.8509 F1=0.8476 QPU/s=2.0125s


RESULTS: 10 RUNS x 5000 SHOTS x 275 SAMPLES
Run    | SV Acc     | SV F1      | Odra Acc   | Odra F1    | QPU Time/sample (s)
----------------------------------------------------------------------------------------------------
1      | 0.8400     | 0.8268     | 0.8509     | 0.8476     | 2.0123
2      | 0.8400     | 0.8268     | 0.8582     | 0.8561     | 2.0124
3      | 0.8400     | 0.8268     | 0.8545     | 0.8507     | 2.0125
4      | 0.8400     | 0.8268     | 0.8618     | 0.8603     | 2.0124
5      | 0.8400     | 

## IQM Job Timing Distribution
Visualize how time is distributed across different components of IQM job execution (QPU, compilation, queue, network, overhead).

In [ ]:
# ========================================================
# IQM Job Timing Distribution Analysis
# ========================================================

print("\n" + "=" * 70)
print("IQM JOB TIMING BREAKDOWN")
print("=" * 70)

# Select which shot count configuration to analyze
print("\nAvailable shot counts:", [r['shots'] for r in hw_results if r['accuracy'] is not None])
selected_shots = int(input("Enter shot count to analyze timing (or press Enter for last): ") or hw_results[-1]['shots'])

# Find the corresponding estimator from the last evaluation loop
# Note: We need to re-run evaluation or store estimators to access timing data
# For now, we'll use the last hw_estimator from the loop

if not hasattr(hw_estimator, 'timestamp_history') or not hw_estimator.timestamp_history:
    print("⚠️  No timing data available. Run the evaluation first.")
else:
    # Extract timing data
    qpu_times = []
    compile_times = []
    queue_times = []
    network_times = []
    
    for t in hw_estimator.timestamp_history:
        ts = t['raw_timestamps']
        
        # QPU execution time
        if ts.get('execution_started') and ts.get('execution_ended'):
            qpu_times.append((ts['execution_ended'] - ts['execution_started']).total_seconds())
        
        # Compilation time
        if ts.get('compilation_started') and ts.get('compilation_ended'):
            compile_times.append((ts['compilation_ended'] - ts['compilation_started']).total_seconds())
        
        # Queue time (waiting for QPU)
        if ts.get('pending_execution') and ts.get('execution_started'):
            queue_times.append((ts['execution_started'] - ts['pending_execution']).total_seconds())
        
        # Network time (upload + download)
        net_time = 0
        if ts.get('created') and ts.get('pending_compilation'):
            net_time += (ts['pending_compilation'] - ts['created']).total_seconds()
        if ts.get('ready') and ts.get('completed'):
            net_time += (ts['completed'] - ts['ready']).total_seconds()
        network_times.append(net_time)
    
    # Calculate totals
    total_qpu = sum(qpu_times)
    total_compile = sum(compile_times)
    total_queue = sum(queue_times)
    total_network = sum(network_times)
    total_job = sum(t['job_time_total'] for t in hw_estimator.timestamp_history)
    total_other = total_job - (total_qpu + total_compile + total_queue + total_network)
    
    # Print summary
    print(f"\nNumber of jobs: {len(hw_estimator.timestamp_history)}")
    print(f"Total job time: {total_job:.3f}s")
    print("\nTime Breakdown:")
    print(f"  QPU Execution:    {total_qpu*1000:8.2f} ms  ({100*total_qpu/total_job:5.1f}%)")
    print(f"  Compilation:      {total_compile*1000:8.2f} ms  ({100*total_compile/total_job:5.1f}%)")
    print(f"  Queue (waiting):  {total_queue*1000:8.2f} ms  ({100*total_queue/total_job:5.1f}%)")
    print(f"  Network (I/O):    {total_network*1000:8.2f} ms  ({100*total_network/total_job:5.1f}%)")
    print(f"  Other (overhead): {total_other*1000:8.2f} ms  ({100*total_other/total_job:5.1f}%)")
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Pie chart
    labels = ['QPU Execution', 'Compilation', 'Queue', 'Network', 'Other']
    sizes = [total_qpu, total_compile, total_queue, total_network, total_other]
    colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4', '#dfe6e9']
    explode = (0.1, 0, 0, 0, 0)  # Emphasize QPU
    
    ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
            shadow=True, startangle=90)
    ax1.set_title(f'IQM Job Time Distribution\n({selected_shots} shots, {len(hw_estimator.timestamp_history)} jobs)', 
                  fontsize=14, fontweight='bold')
    
    # Bar chart
    ax2.bar(labels, [s*1000 for s in sizes], color=colors)
    ax2.set_ylabel('Time (ms)', fontsize=12)
    ax2.set_title('Time Breakdown by Component', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "=" * 70)

---
# Stability Analysis

## 10-Run Accuracy Consistency Test
Results of 10 independent evaluation runs (500 shots, 275 samples each) to verify model stability on Odra 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# 10 runs: 500 shots, 275 samples each
runs = list(range(1, 11))
statevector_acc = [0.8436, 0.8364, 0.8291, 0.8327, 0.8364, 0.8364, 0.8436, 0.8400, 0.8327, 0.8436]
odra_acc =        [0.8545, 0.8473, 0.8436, 0.8473, 0.8509, 0.8509, 0.8509, 0.8509, 0.8509, 0.8473]
qpu_time =        [0.2022, 0.2022, 0.2022, 0.2022, 0.2021, 0.2022, 0.2022, 0.2022, 0.2021, 0.2021]

# Summary stats
odra_mean = np.mean(odra_acc)
odra_std = np.std(odra_acc)
sv_mean = np.mean(statevector_acc)
sv_std = np.std(statevector_acc)

# Print table
print('=' * 75)
print('10-RUN STABILITY TEST  |  500 shots  |  275 samples  |  Odra 5 QPU')
print('=' * 75)
print(f\"{'Run':<6} | {'Statevector Acc':<18} | {'Odra 500 shots Acc':<20} | {'QPU Time/sample (s)'}\")
print('-' * 75)
for i in range(10):
    print(f\"{runs[i]:<6} | {statevector_acc[i]:<18.4f} | {odra_acc[i]:<20.4f} | {qpu_time[i]:.4f}\")
print('-' * 75)
print(f\"{'Mean':<6} | {sv_mean:<18.4f} | {odra_mean:<20.4f} | {np.mean(qpu_time):.4f}\")
print(f\"{'Std':<6} | {sv_std:<18.4f} | {odra_std:<20.4f} | {np.std(qpu_time):.4f}\")
print('=' * 75)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(runs, odra_acc, 'o-', color='#2196F3', linewidth=2.5, markersize=9,
        label=f'Odra 5 (mean={odra_mean:.4f}, std={odra_std:.4f})', zorder=3)
ax.plot(runs, statevector_acc, 's--', color='#FF5722', linewidth=2, markersize=8,
        label=f'Statevector (mean={sv_mean:.4f}, std={sv_std:.4f})', zorder=3)

ax.axhline(y=odra_mean, color='#2196F3', linestyle=':', alpha=0.5, linewidth=1.5)
ax.fill_between(runs, odra_mean - odra_std, odra_mean + odra_std, color='#2196F3', alpha=0.1)

ax.axhline(y=sv_mean, color='#FF5722', linestyle=':', alpha=0.5, linewidth=1.5)
ax.fill_between(runs, sv_mean - sv_std, sv_mean + sv_std, color='#FF5722', alpha=0.1)

ax.set_xlabel('Run', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Accuracy Stability Across 10 Runs on Odra 5\n(500 shots, 275 samples per run)',
             fontsize=14, fontweight='bold')
ax.set_xticks(runs)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
ax.set_ylim([0.82, 0.87])
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()